In [9]:
import utils
import numpy as np
import pandas as pd
import os
from scipy.stats import chisquare

import matplotlib.pyplot as plt
import seaborn as sns # Loads in the theme
sns.set_theme()

pd.set_option('future.no_silent_downcasting', True)
#pd.set_option('display.max_rows', None)

In [10]:
# Make output folders
output_dir = os.path.join('..', '..','pre-processed', 'modelling_data')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [11]:
# Preparation
experiment_output_path = os.path.join('..', '..', '..', 'phd_1_task', 'experiment_output', 'behavioural_files')
#experiment_output_path = os.path.join('..', '..', '..', 'phd_1_task', 'experiment_output_verena')

In [12]:
unique_subject_ids = list(range(901, 950))
unique_sessions = [1]

recent_results = utils.analysis_functions.list_files_with_date_and_subject_id(experiment_output_path, unique_subject_ids, unique_sessions)
print(recent_results)

sub-916 did not complete session-01
sub-917 did not complete session-01
sub-918 did not complete session-01
sub-919 did not complete session-01
sub-920 did not complete session-01
sub-921 did not complete session-01
sub-922 did not complete session-01
sub-923 did not complete session-01
sub-924 did not complete session-01
sub-925 did not complete session-01
sub-926 did not complete session-01
sub-927 did not complete session-01
sub-928 did not complete session-01
sub-929 did not complete session-01
sub-930 did not complete session-01
sub-931 did not complete session-01
sub-932 did not complete session-01
sub-933 did not complete session-01
sub-934 did not complete session-01
sub-935 did not complete session-01
sub-936 did not complete session-01
sub-937 did not complete session-01
sub-938 did not complete session-01
sub-939 did not complete session-01
sub-940 did not complete session-01
sub-941 did not complete session-01
sub-942 did not complete session-01
sub-943 did not complete ses

In [13]:
# Opens all the files and combines them into one dataframe
combined_results_dataframe = utils.analysis_functions.combine_result_files(recent_results)
if 'verena' in experiment_output_path:
    combined_dataframe = combined_results_dataframe[~combined_results_dataframe['stimuli_type'].isin([2, 4])]
    
print(combined_results_dataframe)

       trial  emotional_cue_time response objectively_correct  \
0          0        1.719394e+09      NaN                Late   
1          1        1.719394e+09       up                Even   
2          2        1.719394e+09       up                Even   
3          3        1.719394e+09     down                Even   
4          4        1.719394e+09     down                Even   
...      ...                 ...      ...                 ...   
10195    675        1.720007e+09       up                True   
10196    676        1.720007e+09     down                True   
10197    677        1.720007e+09       up                True   
10198    678        1.720007e+09     down                True   
10199    679        1.720007e+09       up                True   

      subjectively_correct  response_time   RT_s  stimuli_type  iti  \
0                     Late   1.719394e+09  1.202             1    3   
1                     True   1.719394e+09  1.201             3    2   
2     

# Check congruency, valence and volatility

In [14]:
# Check a block's congruency
combined_results_dataframe['congruency'] = combined_results_dataframe.apply(utils.analysis_functions.check_congruency, args=(True,), axis=1)

# Check a block's valence
combined_results_dataframe['valence'] = combined_results_dataframe.apply(utils.analysis_functions.check_valence, args=(True,), axis=1)

# Check a block's volatility
combined_results_dataframe = combined_results_dataframe[combined_results_dataframe['probability_condition'] != 50]
combined_results_dataframe = combined_results_dataframe.groupby('stimuli_type', group_keys=False).apply(lambda x: utils.analysis_functions.check_volatility(x, True))
combined_results_dataframe = combined_results_dataframe.sort_values(by=['subject_id', 'session', 'stimuli_type', 'trial'], ascending=[True, True, True, True])
combined_results_dataframe.reset_index(drop=True, inplace=True)
print(combined_results_dataframe[['subject_id', 'session', 'congruency', 'valence', 'temp_volatility', 'volatility']])
combined_results_dataframe = combined_results_dataframe.drop(columns=['temp_volatility'])

print(combined_results_dataframe['valence'].unique())
print(combined_results_dataframe['congruency'].unique())
print(combined_results_dataframe['volatility'].unique())

      subject_id     session  congruency  valence temp_volatility volatility
0        sub-901  session-01           1        0               1          1
1        sub-901  session-01           1        0               1          1
2        sub-901  session-01           1        0               1          1
3        sub-901  session-01           1        0               1          1
4        sub-901  session-01           1        0               1          1
...          ...         ...         ...      ...             ...        ...
10071    sub-915  session-01           1        1                          0
10072    sub-915  session-01           1        1                          0
10073    sub-915  session-01           1        1                          0
10074    sub-915  session-01           1        1                          0
10075    sub-915  session-01           1        1                          0

[10076 rows x 6 columns]
[0 1]
[1 0]
[1 0]


/var/folders/bh/ld02p1_550d75y0x32dmpb540000gp/T/ipykernel_2820/1720476028.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_results_dataframe = combined_results_dataframe.groupby('stimuli_type', group_keys=False).apply(lambda x: utils.analysis_functions.check_volatility(x, True))


# Recode existing variables

In [15]:
combined_results_dataframe = combined_results_dataframe.dropna(subset=['response'])

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'up': 1, 'down': 0},
                                                                                    'response',
                                                                                    'response_int')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'True': 1, 'False': 0},
                                                                                    'subjectively_correct',
                                                                                    'subjectively_correct_int')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'True': 1, 'False': 0},
                                                                                    'objectively_correct',
                                                                                    'objectively_correct_int')

combined_results_dataframe['subject_id'] = combined_results_dataframe['subject_id'].str.replace(r'\D', '', regex=True)

combined_results_dataframe['session'] = combined_results_dataframe['session'].str.replace(r'\D', '', regex=True)

# Now export the data

In [16]:
combined_results_dataframe.to_csv(f'{output_dir}/combined_dataframe_binary.csv', sep=';', index_label='index')